In [1]:
import math
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Subset, ConcatDataset

from torchvision import datasets, transforms

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from IPython.display import clear_output

In [2]:
class TimeEmbed(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, dim),
            nn.SiLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, t):
        return self.mlp(t[:, None])

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_emb = nn.Linear(t_dim, out_ch)

    def forward(self, x, t):
        h = F.silu(self.conv1(x))
        h = h + self.time_emb(t)[:, :, None, None]
        h = F.silu(self.conv2(h))
        return h

class DoubleStandardClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        base = 32
        t_dim = 32

        self.time_embed = TimeEmbed(t_dim)

        self.down1 = Block(1, base, t_dim)
        self.down2 = Block(base, base * 2, t_dim)
        self.down3 = Block(base * 2, base * 2, t_dim)
        self.pool = nn.MaxPool2d(2)

        self.mid = Block(base * 2, base * 2, t_dim)

        self.out = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(base * 2, 2)
        )
        
    def forward(self, x, t):
        t = self.time_embed(t)

        d1 = self.down1(x, t)
        d2 = self.down2(self.pool(d1), t)
        d3 = self.down3(self.pool(d2), t)

        m = self.mid(d3, t)

        return self.out(m)

class SingleStandardClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        base = 22
        t_dim = 32

        self.time_embed = TimeEmbed(t_dim)

        self.down1 = Block(1, base, t_dim)
        self.down2 = Block(base, base * 2, t_dim)
        self.down3 = Block(base * 2, base * 2, t_dim)
        self.pool = nn.MaxPool2d(2)

        self.mid = Block(base * 2, base * 2, t_dim)

        self.out = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(base * 2, 2)
        )
        
    def forward(self, x, t):
        t = self.time_embed(t)

        d1 = self.down1(x, t)
        d2 = self.down2(self.pool(d1), t)
        d3 = self.down3(self.pool(d2), t)

        m = self.mid(d3, t)

        return self.out(m)

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, groups=8):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),  # GroupNorm is standard for Diffusion
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, time_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        self.input = DoubleConv(1, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))

        self.bottleneck = DoubleConv(256, 512)

        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(384, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(192, 128)

        self.output = nn.Conv2d(128, 1, kernel_size=1)

        self.emb_down1 = nn.Linear(time_dim, 128)
        self.emb_down2 = nn.Linear(time_dim, 256)
        self.emb_up1 = nn.Linear(time_dim, 256)
        self.emb_up2 = nn.Linear(time_dim, 128)

    def forward(self, x, t):
        t = t.view(-1, 1) 
        emb = self.time_mlp(t)
        
        x1 = self.input(x)
        
        x2 = self.down1(x1)
        x2 = x2 + self.emb_down1(emb)[:, :, None, None]
        
        x3 = self.down2(x2)
        x3 = x3 + self.emb_down2(emb)[:, :, None, None]

        x_mid = self.bottleneck(x3)

        x_up1 = self.up1(x_mid)
        x_up1 = torch.cat([x_up1, x2], dim=1)
        x_up1 = self.conv_up1(x_up1) 
        x_up1 = x_up1 + self.emb_up1(emb)[:, :, None, None]
        
        x_up2 = self.up2(x_up1)
        x_up2 = torch.cat([x_up2, x1], dim=1)
        x_up2 = self.conv_up2(x_up2)
        x_up2 = x_up2 + self.emb_up2(emb)[:, :, None, None]

        return self.output(x_up2)

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.GroupNorm(16, 64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.GroupNorm(16, 128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.GroupNorm(16, 256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

        self.fc = nn.Linear(256, 30)

    def forward(self, x):
        x = self.features(x)
        return self.fc(x)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

patternClassifier = SingleStandardClassifier().to(device)
clothesClassifier = SingleStandardClassifier().to(device)
doubleClassifier = DoubleStandardClassifier().to(device)
cls = Classifier().to(device)
uNet = UNet().to(device)

patternClassifier.eval()
clothesClassifier.eval()
doubleClassifier.eval()
cls.eval()
uNet.eval()

# doubleClassifier.load_state_dict(torch.load("single_dre.pth", map_location=device))
# clothesClassifier.load_state_dict(torch.load("duo_type_dre.pth", map_location=device))
# patternClassifier.load_state_dict(torch.load("duo_patt_dre.pth", map_location=device))

cls.load_state_dict(torch.load("fashion_pattern_mnist_cnn.pth", map_location=device))
uNet.load_state_dict(torch.load("diffusion.pth", map_location=device))

img_size = 28
batch_size = 128
epochs = 100
timesteps = 500

beta = torch.linspace(1e-4, 0.02, timesteps).to(device)
alpha = 1. - beta
alpha_bar = torch.cumprod(alpha, dim=0)

In [29]:
n_samples = 500
addition = []
s_values = [0, 0.5, 1, 1.5, 2]

for s in s_values:
# for j in range(1):
    # doubleClassifier.load_state_dict(torch.load(f"Training DRES/CloPat DRE/best_CloPat_{i}.pt", map_location=device))
    clothesClassifier.load_state_dict(torch.load(f"Training DRES/clothes DRE/best_clothes_{0}.pt", map_location=device))
    patternClassifier.load_state_dict(torch.load(f"Training DRES/pattern DRE/best_pattern_{0}.pt", map_location=device))

    addition_outputs = []
    with torch.no_grad():
        xt = torch.randn(n_samples, 1, img_size, img_size).to(device)
        
        for i in reversed(range(timesteps)):
            t = torch.tensor([i] * n_samples, device=device)
            
            eps_pred = uNet(xt, t.float() / timesteps)
            
            alpha_t = alpha[i]
            alpha_bar_t = alpha_bar[i]
            beta_t = beta[i]
    
            mean = (xt - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * eps_pred) / torch.sqrt(alpha_t)
    
            with torch.enable_grad():
                dre_xt = xt.detach().clone().requires_grad_(True)
                
                logits = clothesClassifier(dre_xt, t.float() / timesteps) 
                log_probs = F.log_softmax(logits, dim=1)
                
                r = log_probs[:, 1] - log_probs[:, 0]
                
                pattern_guide_score = torch.autograd.grad(
                    outputs=r.sum(),
                    inputs=dre_xt
                )[0]

            with torch.enable_grad():
                dre_xt = xt.detach().clone().requires_grad_(True)
                
                logits = patternClassifier(dre_xt, t.float() / timesteps)
                log_probs = F.log_softmax(logits, dim=1)
                
                r = log_probs[:, 1] - log_probs[:, 0]
                
                shape_guide_score = torch.autograd.grad(
                    outputs=r.sum(),
                    inputs=dre_xt
                )[0]
            mean = mean + s * beta_t * (pattern_guide_score + shape_guide_score)
            
            if i > 0:
                noise = torch.randn_like(xt)
                xt = mean + torch.sqrt(beta_t) * noise
            else:
                xt = mean
        print("s: {} complete", s)
        addition_outputs.append((s, xt))
    addition.append(addition_outputs)

s: {} complete 0
s: {} complete 0.5
s: {} complete 1
s: {} complete 1.5
s: {} complete 2


In [30]:
n_samples = 500
double = []
s_values = [0, 0.5, 1, 1.5, 2]

for s in s_values:
    
    doubleClassifier.load_state_dict(torch.load(f"Training DRES/CloPat DRE/best_CloPat_{0}.pt", map_location=device))
    
    # clothesClassifier.load_state_dict(torch.load(f"Training DRES/clothes DRE/best_clothes_{j}.pt", map_location=device))
    # patternClassifier.load_state_dict(torch.load(f"Training DRES/pattern DRE/best_pattern_{j}.pt", map_location=device))
    double_outputs = []
    with torch.no_grad():
        xt = torch.randn(n_samples, 1, img_size, img_size).to(device)
        
        for i in reversed(range(timesteps)):
            t = torch.tensor([i] * n_samples, device=device)
            
            eps_pred = uNet(xt, t.float() / timesteps)
            
            alpha_t = alpha[i]
            alpha_bar_t = alpha_bar[i]
            beta_t = beta[i]
    
            mean = (xt - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * eps_pred) / torch.sqrt(alpha_t)
    
            with torch.enable_grad():
                dre_xt = xt.detach().clone().requires_grad_(True)
                
                logits = doubleClassifier(dre_xt, t.float() / timesteps) 
                log_probs = F.log_softmax(logits, dim=1)
                
                r = log_probs[:, 1] - log_probs[:, 0]
                
                double_guide_score = torch.autograd.grad(
                    outputs=r.sum(),
                    inputs=dre_xt
                )[0]

            mean = mean + s * beta_t * double_guide_score
            
            if i > 0:
                noise = torch.randn_like(xt)
                xt = mean + torch.sqrt(beta_t) * noise
            else:
                xt = mean
        print("s: {} complete", s)
        double_outputs.append((s, xt))
    double.append(double_outputs)

s: {} complete 0
s: {} complete 0.5
s: {} complete 1
s: {} complete 1.5
s: {} complete 2


In [34]:
def add_wave(img):
    amplitude, frequency, height, thickness, phase = random.randint(4, 10), random.randint(1, 4), random.randint(3, 20), random.randint(1, 3), random.uniform(0, 2 * math.pi)
    
    i = torch.arange(28)
    j = torch.arange(28)
    curve = height + amplitude * torch.cos(2 * math.pi * frequency * j / 28 + phase)
    
    rows = curve.round().long()
    x = torch.ones(28, 28)
    for d in range(-thickness, thickness + 1):
        r = (rows + d).clamp(0, 27)
        x[r, j] = 0.5
    return img * x

test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=None
)

test_base  = test_dataset.data.float().clone()[(test_dataset.targets == 5) | (test_dataset.targets == 7) | (test_dataset.targets == 9)] / 255.0

waves_test = []
for i in range(test_base.shape[0]):
    waves_test.append(add_wave(test_base[i]))
waves_test = torch.stack(waves_test)

real_images = transforms.Normalize((0.5,), (0.5,))(waves_test).unsqueeze(1)

In [35]:
def knn_radius(features, k=3):
    dists = torch.cdist(features, features)
    dists.fill_diagonal_(float("inf"))
    return dists.kthvalue(k, dim=1).values


def precision(real_feats, fake_feats, r_real):
    d = torch.cdist(fake_feats, real_feats)
    return (d <= r_real.unsqueeze(0)).any(dim=1).float().mean().item()


def recall(real_feats, fake_feats, r_fake):
    d = torch.cdist(real_feats, fake_feats)
    return (d <= r_fake.unsqueeze(0)).any(dim=1).float().mean().item()

In [36]:
experiments_double = []
cls.eval()
for experiment in double:
    metrics = []
    perm = torch.randperm(real_images.shape[0])[:50]
    for s, fake_images in experiment:
        real_feats = cls.features(real_images[perm].cuda())
        fake_feats = cls.features(fake_images)
        
        r_real = knn_radius(real_feats)
        r_fake = knn_radius(fake_feats)
        
        p = precision(real_feats, fake_feats, r_real)
        r = recall(real_feats, fake_feats, r_fake)
    
        # print(f"Guidance: {s}, Precision: {p:.4f}, Recall: {r:.4f}")
        metrics.append((s, p, r, (2 * p * r) / (p + r)))
    experiments_double.append(metrics)

experiments_addition = []
cls.eval()
for experiment in addition:
    metrics = []
    perm = torch.randperm(real_images.shape[0])[:50]
    for s, fake_images in experiment:
        real_feats = cls.features(real_images[perm].cuda())
        fake_feats = cls.features(fake_images)
        
        r_real = knn_radius(real_feats)
        r_fake = knn_radius(fake_feats)
        
        p = precision(real_feats, fake_feats, r_real)
        r = recall(real_feats, fake_feats, r_fake)
    
        # print(f"Guidance: {s}, Precision: {p:.4f}, Recall: {r:.4f}")
        metrics.append((s, p, r, (2 * p * r) / (p + r)))
    experiments_addition.append(metrics)

In [41]:
# experiments_addition, experiments_double

([[(0, 0.12200000882148743, 0.8799999952316284, 0.2142914306325221)],
  [(0.5, 0.3720000088214874, 0.9799999594688416, 0.5392899439612718)],
  [(1, 0.5760000348091125, 0.85999995470047, 0.6899164449332336)],
  [(1.5, 0.5520000457763672, 0.7599999904632568, 0.6395122224663341)],
  [(2, 0.5740000009536743, 0.7999999523162842, 0.6684133755603768)]],
 [[(0, 0.19600000977516174, 0.9199999570846558, 0.3231541325025706)],
  [(0.5, 0.38600000739097595, 0.9399999976158142, 0.547269991941461)],
  [(1, 0.3800000250339508, 0.9799999594688416, 0.5476470784925453)],
  [(1.5, 0.4820000231266022, 0.9599999785423279, 0.6417753277717068)],
  [(2, 0.492000013589859, 0.9199999570846558, 0.6411331455936128)]])

In [42]:
# steps = torch.tensor(experiments_double)[0,:,0]
values = torch.tensor(experiments_double)[:,:,1:]
n = values.shape[0]
mean_exp = values.mean(dim=0)
std_exp = values.std(dim=0, unbiased=True)
sem_exp = std_exp / (n ** 0.5) * 2.776

double_data = steps, mean_exp[:,0], sem_exp[:,0]

# steps = torch.tensor(experiments_addition)[0,:,0]
values = torch.tensor(experiments_addition)[:,:,1:]
n = values.shape[0]
mean_exp = values.mean(dim=0)
std_exp = values.std(dim=0, unbiased=True)
sem_exp = std_exp / (n ** 0.5) * 2.776

addition_data = steps, mean_exp[:,0], sem_exp[:,0]


In [43]:
addition_data

(tensor([0.]), tensor([0.4392]), tensor([0.2442]))